# Validate and Load Aged Care Facility Availability

This notebook reads the facility availability file, checks that each label ID matches the expected label name,
and inserts the validated data into Postgres.


In [ ]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path("../input/facility_cluster_labels_updated.csv")
INPUT_FILE

PosixPath('input/facility_cluster_labels_updated.csv')

In [2]:
df = pd.read_csv(INPUT_FILE)

for col in ["residential_label_id", "home_care_label_id", "restorative_care_label_id"]:
    print(col, sorted(pd.to_numeric(df[col], errors="coerce").dropna().astype(int).unique().tolist()))

residential_label_id [1, 2, 3, 4, 5]
home_care_label_id [0, 1, 3, 4, 5]
restorative_care_label_id [0, 1, 2, 3, 4, 5]


In [3]:
raw_df = pd.read_csv(INPUT_FILE)
raw_df.head()

,facility_id,residential_label_id,residential_label_name,home_care_label_id,home_care_label_name,restorative_care_label_id,restorative_care_label_name
0,001085ed-daf6-48a5-9029-ac890675c5c1,1,Likely Available,1,Likely Available,0,Does Not Provide This Service
1,001efff7-26d9-46e6-8a51-a8de7efe201e,4,Constrained by Market,0,Does Not Provide This Service,0,Does Not Provide This Service
2,0022f30c-0cc0-4ccd-a59d-4ef20a1409f7,5,Highly Constrained,0,Does Not Provide This Service,0,Does Not Provide This Service
3,00351956-40c3-4b61-9855-9917e3f21967,4,Constrained by Market,0,Does Not Provide This Service,0,Does Not Provide This Service
4,0038e720-09b6-4cef-84cc-3b6f8a634de7,5,Highly Constrained,0,Does Not Provide This Service,0,Does Not Provide This Service


In [4]:
def validate_and_prepare_facility_availability(input_path: Path):
    df = pd.read_csv(input_path).copy()

    expected_columns = [
        "facility_id",
        "residential_label_id",
        "residential_label_name",
        "home_care_label_id",
        "home_care_label_name",
        "restorative_care_label_id",
        "restorative_care_label_name",
    ]

    missing_cols = [c for c in expected_columns if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    text_cols = [
        "facility_id",
        "residential_label_name",
        "home_care_label_name",
        "restorative_care_label_name",
    ]
    for col in text_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    int_cols = [
        "residential_label_id",
        "home_care_label_id",
        "restorative_care_label_id",
    ]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    checks = [
        ("residential_label_id", "residential_label_name"),
        ("home_care_label_id", "home_care_label_name"),
        ("restorative_care_label_id", "restorative_care_label_name"),
    ]

    inconsistent_frames = []

    for id_col, name_col in checks:
        pair_df = (
            df[[id_col, name_col]]
            .dropna()
            .drop_duplicates()
            .groupby(id_col)[name_col]
            .nunique()
            .reset_index(name="name_count")
        )

        bad_ids = pair_df.loc[pair_df["name_count"] > 1, id_col]

        if not bad_ids.empty:
            bad_rows = (
                df[df[id_col].isin(bad_ids)]
                [[id_col, name_col]]
                .drop_duplicates()
                .sort_values([id_col, name_col])
                .copy()
            )
            bad_rows["label_group"] = id_col.replace("_label_id", "")
            inconsistent_frames.append(bad_rows)

    inconsistent_df = (
        pd.concat(inconsistent_frames, ignore_index=True)
        if inconsistent_frames else pd.DataFrame()
    )

    duplicate_facility_ids = (
        df["facility_id"]
        .dropna()
        .value_counts()
        .loc[lambda x: x > 1]
    )

    duplicate_df = df[df["facility_id"].isin(duplicate_facility_ids.index)].copy()

    summary = {
        "rows": len(df),
        "unique_facilities": df["facility_id"].nunique(dropna=True),
        "duplicate_facility_id_count": int(len(duplicate_facility_ids)),
        "duplicate_facility_rows": int(len(duplicate_df)),
        "inconsistent_label_mappings": int(len(inconsistent_df)),
    }

    if not inconsistent_df.empty:
        raise ValueError(
            "Some label IDs map to multiple label names. Review inconsistent_df before inserting."
        )

    if not duplicate_df.empty:
        raise ValueError(
            "Duplicate facility_id values found. Review duplicate_df before inserting."
        )

    df = df.dropna(subset=["facility_id"]).copy()

    return df.reset_index(drop=True), summary, inconsistent_df, duplicate_df

In [5]:
clean_df, summary, inconsistent_df, duplicate_df = validate_and_prepare_facility_availability(INPUT_FILE)
print(summary)

{'rows': 3015, 'unique_facilities': 3015, 'duplicate_facility_id_count': 0, 'duplicate_facility_rows': 0, 'inconsistent_label_mappings': 0}


In [6]:

print(f"Data types:\n{clean_df.dtypes}")

Data types:
facility_id                    string
residential_label_id            Int64
residential_label_name         string
home_care_label_id              Int64
home_care_label_name           string
restorative_care_label_id       Int64
restorative_care_label_name    string
dtype: object


# Insert Validated Data into DB

In [7]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/aged_care_facility_availability.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [8]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "facility_id",
    "residential_label_id",
    "residential_label_name",
    "home_care_label_id",
    "home_care_label_name",
    "restorative_care_label_id",
    "restorative_care_label_name",
]

df_insert = df_insert[target_columns].copy()

text_cols = [
    "facility_id",
    "residential_label_name",
    "home_care_label_name",
    "restorative_care_label_name",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

int_cols = [
    "residential_label_id",
    "home_care_label_id",
    "restorative_care_label_id",
]

for col in int_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce").astype("Int64")

df_insert.head()

,facility_id,residential_label_id,residential_label_name,home_care_label_id,home_care_label_name,restorative_care_label_id,restorative_care_label_name
0,001085ed-daf6-48a5-9029-ac890675c5c1,1,Likely Available,1,Likely Available,0,Does Not Provide This Service
1,001efff7-26d9-46e6-8a51-a8de7efe201e,4,Constrained by Market,0,Does Not Provide This Service,0,Does Not Provide This Service
2,0022f30c-0cc0-4ccd-a59d-4ef20a1409f7,5,Highly Constrained,0,Does Not Provide This Service,0,Does Not Provide This Service
3,00351956-40c3-4b61-9855-9917e3f21967,4,Constrained by Market,0,Does Not Provide This Service,0,Does Not Provide This Service
4,0038e720-09b6-4cef-84cc-3b6f8a634de7,5,Highly Constrained,0,Does Not Provide This Service,0,Does Not Provide This Service


In [9]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "aged_care_facility_availability",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into aged_care_facility_availability")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM aged_care_facility_availability"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 3015 rows into aged_care_facility_availability
Rows in table: 3015
